# Praktikum 11.3 Membuat Model Klasifikasi Multi Kelas

In [1]:
# ==========================================================
# SETUP GOOGLE COLAB: GPU & AUTO-DOWNLOAD DATASET
# ==========================================================
import os
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'>>> [AKTIF] Akselerasi GPU Terdeteksi: {gpus[0].name}')
else:
    print('>>> [INFO] Mode CPU aktif. Untuk GPU: Runtime -> Change runtime type -> T4 GPU.')

if not os.path.exists('Dataset 1') or not os.path.exists('Dataset 2') or not os.path.exists('Dataset 3'):
    print('\n>>> Mengunduh dataset dari server CDN...')
    os.system('wget -q https://files.catbox.moe/po8wfa.zip -O dataset_modul11.zip')
    os.system('unzip -q -o dataset_modul11.zip')
    print('>>> [SUKSES] Dataset 1, Dataset 2, dan Dataset 3 siap digunakan!')


>>> [INFO] Mode CPU aktif. Untuk GPU: Runtime -> Change runtime type -> T4 GPU.


### 2) Pada modul train.py lakukan training pada dataset menggunakan CNN hingga menghasilkan sebuah model:

In [2]:
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# ==========================
# DATASET
# ==========================
dataset_dir = "Dataset 3/Dataset" if os.path.exists("Dataset 3/Dataset") else "Dataset"

datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = datagen.flow_from_directory(
    dataset_dir,
    target_size=(224, 224),
    batch_size=2,
    class_mode="categorical"
)

print("Jumlah data :", train_data.samples)
print("Kelas :", train_data.class_indices)

# ==========================
# CNN
# ==========================
model = Sequential([
    Conv2D(32, (3, 3), padding="same", activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(5, activation="softmax")
])

# ==========================
# COMPILE
# ==========================
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================
# TRAINING
# ==========================
model.fit(
    train_data,
    epochs=10
)

# ==========================
# SAVE
# ==========================
model.save("model_makanan.keras")

Found 100 images belonging to 5 classes.


Jumlah data : 100
Kelas : {'Bakwan Sayur': 0, 'Bubur Tinotuan': 1, 'Nasi Beras Merah': 2, 'Plencing Kangkung': 3, 'Wortel': 4}


C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


 1/50 ━━━━━━━━━━━━━━━━━━━━ 43s 880ms/step - accuracy: 0.0000e+00 - loss: 1.6234

 2/50 ━━━━━━━━━━━━━━━━━━━━ 4s 96ms/step - accuracy: 0.0000e+00 - loss: 15.6119 

... (513 outputs left out)

48/50 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - accuracy: 1.0000 - loss: 7.1104e-05

49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step - accuracy: 1.0000 - loss: 6.9707e-05

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step - accuracy: 1.0000 - loss: 6.8545e-05

50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 228ms/step - accuracy: 1.0000 - loss: 6.8545e-05


### 6) Pada modul test.py lakukan pengujian untuk tes0.jpeg dengan memanggil model sebelumnya:

In [3]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import os

# LOAD MODEL
model = load_model("model_makanan.keras")

# DAFTAR KELAS
kelas = {
    0: "Bakwan Sayur",
    1: "Bubur Tinotuan",
    2: "Nasi Beras Merah",
    3: "Plencing Kangkung",
    4: "Wortel"
}

# LOAD GAMBAR
img_path = "tes0.jpeg" if os.path.exists("tes0.jpeg") else "Dataset 3/tes0.jpeg"
img = image.load_img(
    img_path,
    target_size=(224, 224)
)

# Ubah gambar menjadi array
img = image.img_to_array(img)

# Normalisasi
img = img / 255.0

# Tambahkan dimensi batch
img = np.expand_dims(img, axis=0)

# PREDIKSI
hasil = model.predict(img)

# Ambil index dengan nilai terbesar
index = np.argmax(hasil[0])

# Ambil nilai probabilitas
nilai = hasil[0][index]

# HASIL
print("Nilai :", nilai)
print("Index kelas :", index)
print("Kelas :", kelas[index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step


Nilai : 0.99974114
Index kelas : 0
Kelas : Bakwan Sayur


### 7) Dengan cara yang sama lakukan pengujian kembali untuk citra yang lain (tes1.jpeg s/d tes4.jpeg):

In [4]:
for i in range(1, 5):
    fname = f"tes{i}.jpeg" if os.path.exists(f"tes{i}.jpeg") else f"Dataset 3/tes{i}.jpeg"
    print(f"\n================ Pengujian untuk tes{i}.jpeg ================")
    img = image.load_img(fname, target_size=(224, 224))
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    hasil = model.predict(img)
    index = np.argmax(hasil[0])
    nilai = hasil[0][index]

    print("Nilai :", nilai)
    print("Index kelas :", index)
    print("Kelas :", kelas[index])


================ Pengujian untuk tes1.jpeg ================
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


Nilai : 0.61787486
Index kelas : 1
Kelas : Bubur Tinotuan

================ Pengujian untuk tes2.jpeg ================


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


Nilai : 0.8719566
Index kelas : 2
Kelas : Nasi Beras Merah


... (3 outputs left out)

Nilai : 0.99999917
Index kelas : 3
Kelas : Plencing Kangkung

================ Pengujian untuk tes4.jpeg ================


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


Nilai : 0.9999658
Index kelas : 4
Kelas : Wortel
